In [413]:
import pandas as pd

In [414]:
df1 = pd.read_json('../data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json')
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50336 entries, 0 to 50335
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   도메인        50336 non-null  object
 1   카테고리       50336 non-null  object
 2   대화셋일련번호    50336 non-null  object
 3   화자         50336 non-null  object
 4   문장번호       50336 non-null  int64 
 5   고객의도       50336 non-null  object
 6   상담사의도      50336 non-null  object
 7   QA         50336 non-null  object
 8   고객질문(요청)   50336 non-null  object
 9   상담사질문(요청)  50336 non-null  object
 10  고객답변       50336 non-null  object
 11  상담사답변      50336 non-null  object
 12  개체명        50336 non-null  object
 13  용어사전       50336 non-null  object
 14  지식베이스      50336 non-null  object
dtypes: int64(1), object(14)
memory usage: 5.8+ MB


In [415]:
# 결측치의 개수를 확인
df1.isna().sum()

도메인          0
카테고리         0
대화셋일련번호      0
화자           0
문장번호         0
고객의도         0
상담사의도        0
QA           0
고객질문(요청)     0
상담사질문(요청)    0
고객답변         0
상담사답변        0
개체명          0
용어사전         0
지식베이스        0
dtype: int64

In [416]:
df1.head(10)

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까?,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
3,다산콜센터,일반행정 문의,B2240,상담사,4,,지방세납부,Q,,어떤 은행을 이용하고 계십니까?,,,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,5,지방세납부,,A,,,기업은행을 이용하고 있습니다.,,기업은행,기업은행/상호,"기업은행,상호"
5,다산콜센터,일반행정 문의,B2240,상담사,6,,지방세납부,A,,,,그럼 스마트폰에서 기업은행 어플을 설치하시면 납부가 가능합니다.,"스마트폰, 기업은행, 어플, 납부",스마트폰/전자기기/ 기업은행/상호,"기업은행,상호"
6,다산콜센터,일반행정 문의,B2240,고객,7,지방세납부,,Q,은행을 직접방문해도 됩니까?,,,,은행,은행/공공기관,"은행,공공기관"
7,다산콜센터,일반행정 문의,B2240,상담사,8,,지방세납부,A,,,,방문납부도 가능합니다.,,,
8,다산콜센터,일반행정 문의,B2240,고객,9,지방세납부,,Q,은행위치 좀 알 수 있습니까?,,,,은행,은행/공공기관,"은행,공공기관"
9,다산콜센터,일반행정 문의,B2240,상담사,10,,지방세납부,Q,,어느지점으로 안내해드릴까요?,,,,,


### 문제
1. 일반행정 데이터와 대중교통 데이터를 로드
2. 두개의 데이터프레임을 결합(단순한 행 결합)
3. 데이터의 필터링 고객질문에 대한 상담사의 답변이 즉각적으로 오는 데이터들만 필터
4. 질문 중 중복 데이터를 제거
5. 질문들을 모아서 토큰화, 벡터화
5. 그 외의 질문 목륵을 이용하여 코사인 유사도 확인하고 유사 질문과 답변을 출력

In [417]:
df2 = pd.read_json("../data/민원(콜센터) 질의응답_다산콜센터_대중교통 안내_Training.json")
df2.head()

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,대중교통 안내,B2033,고객,1,버스노선,,Q,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,,,,"서울, 가산동, 남대문시장, 버스, 노선",서울/지명/ 가산동/동네/ 남대문시장/지명/ 버스/교통수단,"가산동,교통수단"
1,다산콜센터,대중교통 안내,B2033,상담사,2,,버스노선,Q,,가산동 어디에서 출발하십니까?,,,"가산동, 출발",가산동/동네/ 출발/출발지,"출발,출발지"
2,다산콜센터,대중교통 안내,B2033,고객,3,버스노선,,A,,,가산동 주민센터입니다.,,"가산동, 주민센터",가산동/동네/ 주민센터/공공기관,"주민센터,공공기관"
3,다산콜센터,대중교통 안내,B2033,상담사,4,,버스노선,A,,,,가산동 주민센터에서 남대문시장으로 가는 버스노선은 505번 버스입니다.,"가산동, 주민센터, 남대문시장,버스, 노선",가산동/동네/ 주민센터/공공기관/ 남대문시장/지명/ 버스/교통수단,"주민센터,교통수단"
4,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,,정류장,,정류장


In [418]:
df = pd.concat([df1, df2])
df.shape

(89302, 15)

In [419]:
df.head(10)

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까?,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
3,다산콜센터,일반행정 문의,B2240,상담사,4,,지방세납부,Q,,어떤 은행을 이용하고 계십니까?,,,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,5,지방세납부,,A,,,기업은행을 이용하고 있습니다.,,기업은행,기업은행/상호,"기업은행,상호"
5,다산콜센터,일반행정 문의,B2240,상담사,6,,지방세납부,A,,,,그럼 스마트폰에서 기업은행 어플을 설치하시면 납부가 가능합니다.,"스마트폰, 기업은행, 어플, 납부",스마트폰/전자기기/ 기업은행/상호,"기업은행,상호"
6,다산콜센터,일반행정 문의,B2240,고객,7,지방세납부,,Q,은행을 직접방문해도 됩니까?,,,,은행,은행/공공기관,"은행,공공기관"
7,다산콜센터,일반행정 문의,B2240,상담사,8,,지방세납부,A,,,,방문납부도 가능합니다.,,,
8,다산콜센터,일반행정 문의,B2240,고객,9,지방세납부,,Q,은행위치 좀 알 수 있습니까?,,,,은행,은행/공공기관,"은행,공공기관"
9,다산콜센터,일반행정 문의,B2240,상담사,10,,지방세납부,Q,,어느지점으로 안내해드릴까요?,,,,,


In [420]:
list(df['고객질문(요청)'].index)
list(df['상담사답변'])

['',
 '이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.',
 '',
 '',
 '',
 '그럼 스마트폰에서 기업은행 어플을 설치하시면 납부가 가능합니다.',
 '',
 '방문납부도 가능합니다.',
 '',
 '',
 '',
 '도보로 10분위치에 서울역지점이 있습니다.',
 '',
 '도보로 가시는게 빠를 것 같습니다.',
 '',
 '위택스 사이트에서 납부하실수 있습니다.',
 '',
 'www.wetax.go.kr 입니다.',
 '',
 '이용하시는 은행의 사이트에서도 지방세 납부가 가능합니다.',
 '',
 '간단한 본인확인 후 안내해드리겠습니다.',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '본인확인이 되셨습니다.',
 '',
 '지방세는 22만원으로 조회됩니다.',
 '',
 '온라인으로는 위택스나 은행 홈페이지에서 가능합니다.',
 '',
 'OOOO.OOO.go.kr 입니다.',
 '',
 '',
 '',
 '현재로썬 진행될 예정입니다. ',
 '',
 '네, 철저한 방역수칙하에 진행할 예정입니다. ',
 '',
 '',
 '예매사이트 통해서 환불하실 수 있습니다. ',
 '',
 '공연 일주일 전까진 10% 수수료가 있습니다. ',
 '',
 '네 규정상 어쩔수 없습니다.',
 '',
 '철저한 방역수칙으로 진행하기에 걱정 안하셔도 됩니다. ',
 '',
 '그 부분에 대해서는 개인이 조심을 하셔야기때문에 서울시에서는 따로 보상을 해드리진 않습니다. ',
 '',
 '확산세가 심하면 취소 될 수도 있습니다. ',
 '',
 '네 맞습니다. ',
 '',
 '',
 '죄송하지만 이미 신청기간이 지났습니다. ',
 '',
 '희망두배 청년통장은 신청하실수 있습니다. ',
 '',
 '청년저축계좌랑 비슷한 정책입니다. ',
 '',
 '네 맞습니다. ',
 '',
 '2~3년간 꾸준히 저축하는 서울청년 대상 본인 저축액 100%를 서울시 예산으로 추가 적립해주는 통장입니다.',
 '',
 '저축액을

In [421]:
df.head()

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까?,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
3,다산콜센터,일반행정 문의,B2240,상담사,4,,지방세납부,Q,,어떤 은행을 이용하고 계십니까?,,,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,5,지방세납부,,A,,,기업은행을 이용하고 있습니다.,,기업은행,기업은행/상호,"기업은행,상호"


In [422]:
df_list = []

for data in range(len(df)-1):
    if df.loc[data, '고객질문(요청)'] != '':
        if df.loc[data+1, '상담사답변'] != '':
            df_list.append(df.iloc[data])
            df_list.append(df.iloc[data+1])
df_list

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
cond_b = df["고객질문(요청)"].astype(str).str.strip() != ""

# 다음 행의 d가 비어 있지 않은 행
cond_next_d = df["상담사답변"].shift(-1).astype(str).str.strip() != ""


target_idx = df.index[cond_b & cond_next_d]
rows_to_show = []
for i in target_idx:
    rows_to_show.extend([i, i+1])

result = df.loc[rows_to_show]
result

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"
0,다산콜센터,대중교통 안내,B2033,고객,1,버스노선,,Q,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,,,,"서울, 가산동, 남대문시장, 버스, 노선",서울/지명/ 가산동/동네/ 남대문시장/지명/ 버스/교통수단,"가산동,교통수단"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
1,다산콜센터,대중교통 안내,B2033,상담사,2,,버스노선,Q,,가산동 어디에서 출발하십니까?,,,"가산동, 출발",가산동/동네/ 출발/출발지,"출발,출발지"
6,다산콜센터,일반행정 문의,B2240,고객,7,지방세납부,,Q,은행을 직접방문해도 됩니까?,,,,은행,은행/공공기관,"은행,공공기관"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38937,다산콜센터,대중교통 안내,B25058,상담사,12,,시내버스 시간,A,,,,많이들 궁금해 하더라구요.,궁금,궁금/호기심,"궁금,호기심"
38962,다산콜센터,일반행정 문의,B24967,고객,7,지방세문의,,A,,,근데 컴퓨터할 수 있는 상황이 안되네요,,"컴퓨터, 상황",컴퓨터 - 기계,"상황,컴퓨터-기계"
38962,다산콜센터,대중교통 안내,B25059,고객,17,시내버스 시간,,Q,그럼 순천 시내버스 막차는 몇 시 인가요?,,,,"순천, 시내, 버스, 차, 시","순천/지역, 버스/대중교통, 차/자동차, 시/시간","시내,시간"
38963,다산콜센터,일반행정 문의,B24967,상담사,8,,지방세문의,A,,,,가까운 구청이나 동주민센터에 가시면 바로 발급가능합니다,"구청, 주민센터,발급, 가능","구청 - 건물, 주민센터 - 건물, 발급 - 발행","주민센터,발급-발행"


In [ ]:
 df["고객질문(요청)"].astype(str).str.strip()

0            지방세를 내려면 어떻게 해야됩니까?
1                               
2                  은행 어플에서도 됩니까?
3                               
4                               
                  ...           
38961                           
38962    그럼 순천 시내버스 막차는 몇 시 인가요?
38963                           
38964                           
38965                           
Name: 고객질문(요청), Length: 89302, dtype: object

In [ ]:
df["상담사답변"].shift(-1).astype(str).str.strip()

0               이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.
1                                             
2                                             
3                                             
4          그럼 스마트폰에서 기업은행 어플을 설치하시면 납부가 가능합니다.
                         ...                  
38961                                         
38962    노선마다 다르지만 순천 시내버스는 가장 늦는게 23시55분 입니다.
38963                                         
38964                         그렇군요. 저도 감사드립니다.
38965                                     None
Name: 상담사답변, Length: 89302, dtype: object

In [ ]:
df[df['상담사답변'] != '']

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
5,다산콜센터,일반행정 문의,B2240,상담사,6,,지방세납부,A,,,,그럼 스마트폰에서 기업은행 어플을 설치하시면 납부가 가능합니다.,"스마트폰, 기업은행, 어플, 납부",스마트폰/전자기기/ 기업은행/상호,"기업은행,상호"
7,다산콜센터,일반행정 문의,B2240,상담사,8,,지방세납부,A,,,,방문납부도 가능합니다.,,,
11,다산콜센터,일반행정 문의,B2240,상담사,12,,지방세납부,A,,,,도보로 10분위치에 서울역지점이 있습니다.,"10분, 서울, 역",10분/시간/ 서울/지명/ 역/역사,"서울,역사"
13,다산콜센터,일반행정 문의,B2240,상담사,14,,지방세납부,A,,,,도보로 가시는게 빠를 것 같습니다.,,버스/길안내/교통수단,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38953,다산콜센터,대중교통 안내,B25059,상담사,8,,시내버스 시간,A,,,,아하 알겠습니다. 잠시 찾아보겠습니다.,잠시,잠시/잠깐,"잠시,잠깐"
38955,다산콜센터,대중교통 안내,B25059,상담사,10,,시내버스 시간,A,,,,노선마다 다르지만 순천 시내버스는 가장 빠른게 05시50분 입니다.,"노선, 순천, 시내, 버스, 시, 분","노선/차편, 순천/지역, 버스/대중교통, 시/시간, 분/시간","순천,시간"
38961,다산콜센터,대중교통 안내,B25059,상담사,16,,시내버스 시간,A,,,,네 질문 해 주세요.,질문,질문/문의,"질문,문의"
38963,다산콜센터,대중교통 안내,B25059,상담사,18,,시내버스 시간,A,,,,노선마다 다르지만 순천 시내버스는 가장 늦는게 23시55분 입니다.,"노선, 순천, 시내, 버스, 시, 분","노선/차편, 순천/지역, 버스/대중교통, 시/시간, 분/시간","순천,시간"


In [424]:
df = df.map(
    lambda x : str(x).strip()
)

In [425]:
index = []
for i in range(len(df) -1 ):
    if df.iloc[i]['고객질문(요청)'] != '':
        if df.iloc[i+1]['상담사답변'] != '':
            index.append(i)
            index.append(i+1)
index 


[0,
 1,
 6,
 7,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 34,
 35,
 36,
 37,
 40,
 41,
 42,
 43,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 124,
 125,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 210,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 219,
 222,
 223,
 229,
 230,
 231,
 232,
 233,
 234,
 235,
 236,
 237,
 238,
 242,
 243,
 244,
 245,
 246,
 247,
 249,
 250,
 251,
 2

In [426]:
data = df.iloc[index]
data.reset_index(drop=True, inplace=True)

In [427]:
data['고객질문(요청)'].value_counts()

고객질문(요청)
                             25437
카드결제와 현금결제할 때 요금 차이가 있나요?       79
시간은 얼마나 걸려요?                    55
버스요금은 얼마인가요?                    46
시간은 얼마나 걸리는교?                   43
                             ...  
그건 뭡니까?                          1
서울시민 대상입니까?                      1
얼마를 추가 적립해줍니까?                   1
모집인원은 어떻게 됩니까?                   1
지폐도 받나요?                         1
Name: count, Length: 18953, dtype: int64

In [428]:
data['상담사답변'] = data.shift(-1)['상담사답변']
data = data.drop_duplicates('고객질문(요청)')


C:\Users\student\AppData\Local\Temp\ipykernel_6792\1734240071.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['상담사답변'] = data.shift(-1)['상담사답변']


In [429]:
data.drop(1, inplace=True)

In [430]:
data

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,지방세,지방세/세금,"지방세,세금"
2,다산콜센터,일반행정 문의,B2240,고객,7,지방세납부,,Q,은행을 직접방문해도 됩니까?,,,방문납부도 가능합니다.,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,13,지방세납부,,Q,버스로 가는 방법도 있습니까?,,,도보로 가시는게 빠를 것 같습니다.,버스,버스/교통수단,"버스,교통수단"
6,다산콜센터,일반행정 문의,B2240,고객,15,지방세납부,,Q,다른 납부방법도 있습니까?,,,위택스 사이트에서 납부하실수 있습니다.,,납부방법/지방세,
8,다산콜센터,일반행정 문의,B2240,고객,17,지방세납부,,Q,사이트 주소가 어떻게 됩니까?,,,www.wetax.go.kr 입니다.,사이트,납부방법/지방세/ 위텍스/ 납부,"사이트,납부"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50876,다산콜센터,대중교통 안내,B25056,고객,17,시내버스 시간,,Q,그럼 여수 시내버스 막차는 몇 시 인가요?,,,노선마다 다르지만 여수 시내버스는 가장 늦는게 23시45분 입니다.,"여수, 시내, 버스, 차, 시","여수/지역, 버스/대중교통, 차/자동차, 시/시간","시내,시간"
50878,다산콜센터,대중교통 안내,B25057,고객,7,지하철 여부,,Q,혹시 평택에 지하철 있나요?,,,네 잠시만요.,"평택, 지하철","평택/지역, 지하철/대중교통","지하철,대중교통"
50880,다산콜센터,대중교통 안내,B25058,고객,5,시내버스 시간,,Q,익산 시내버스 첫차가 몇 시 인가요?,,,노선마다 다르지만 익산 시내버스는 가장 빠른게 05시00분 입니다.,"익산, 시내, 버스, 차, 시","익산/지역, 버스/대중교통, 차/자동차, 시/시간","시내,시간"
50886,다산콜센터,대중교통 안내,B25058,고객,11,시내버스 시간,,Q,그럼 익산 시내버스 막차는 몇 시 인가요?,,,많이들 궁금해 하더라구요.,"익산, 시내, 버스, 차, 시","익산/지역, 버스/대중교통, 차/자동차, 시/시간","시내,시간"


In [ ]:
data.reset_index(drop=True, inplace=True)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from konlpy.tag import Komoran

In [ ]:
data['고객질문(요청)'] != ''

0        True
1        True
2        True
3        True
4        True
         ... 
19048    True
19049    True
19050    True
19051    True
19052    True
Name: 고객질문(요청), Length: 19053, dtype: bool

In [ ]:
data['고객질문(요청)']

0            지방세를 내려면 어떻게 해야됩니까?
1                은행을 직접방문해도 됩니까?
2               버스로 가는 방법도 있습니까?
3                 다른 납부방법도 있습니까?
4               사이트 주소가 어떻게 됩니까?
                  ...           
19048    그럼 여수 시내버스 막차는 몇 시 인가요?
19049            혹시 평택에 지하철 있나요?
19050       익산 시내버스 첫차가 몇 시 인가요?
19051    그럼 익산 시내버스 막차는 몇 시 인가요?
19052    그럼 순천 시내버스 막차는 몇 시 인가요?
Name: 고객질문(요청), Length: 19053, dtype: object

In [ ]:
question = list(data['고객질문(요청)'])

In [ ]:
question

['지방세를 내려면 어떻게 해야됩니까?',
 '은행을 직접방문해도 됩니까?',
 '버스로 가는 방법도 있습니까?',
 '다른 납부방법도 있습니까?',
 '사이트 주소가 어떻게 됩니까?',
 '다른곳에서는 납부할수 없습니까?',
 '지방세는 조회할 수 있습니까?',
 '어떻게 납부합니까?',
 '서울시주최 페스티벌 예매해놨는데 예정대로 진행됩니까?',
 '코로나로 다른 축제들은 취소됐는데 이건 취소 안됩니까?',
 '환불할수 있습니까?',
 '환불수수료 있습니까?',
 '코로나때문에 못가는건데도 수수료 내야합니까?',
 '참석했다가 코로나 걸리면 보상해줍니까?',
 '그래도 걸리면 보상해줍니까?',
 '확산세가 더 강해지면 취소될 가능성 있습니까?',
 '그럼 전액 환불되는겁니까?',
 '청년저축계좌 지금 신청할 수 있습니까?',
 '그럼 다른 정책은 없습니까?',
 '그건 뭡니까?',
 '서울시민 대상입니까?',
 '얼마를 추가 적립해줍니까?',
 '모집인원은 어떻게 됩니까?',
 '신청기간은 언제까지입니까?',
 '네 주소지가 서울이면 되는겁니까?',
 '그럼 총 어느정도 혜택을 받을 수 있습니까?',
 '3년동안 계속 10만원씩 해야합니까?',
 '일도 쉬면 안됩니까?',
 '또 다른 조건 있습니까?',
 '저희가 작년에 결혼했는데 신혼부부 조건이 맞습니까?',
 '대출 상품이 뭐가 있습니까?',
 '어떤 은행에서 받을 수 있습니까?',
 '버팀목 대출 받으려면 조건이 있습니까?',
 '서울소재 주택만 가능한겁니까?',
 '대출한도는 어떻게 됩니까?',
 '100% 다 대출받을 수 있습니까?',
 '그럼 나머지 10%에 대해서는 그냥 대출을 받을 수 있습니까?',
 '그럼 그 한정된 곳에서 집을 찾아야합니까?',
 '대출 한도는 어떻게 됩니까?',
 '금리는 얼마입니까?',
 '그렇게 낮은편이 아닌데 다른 혜택은 없습니까?',
 '그게 뭡니까?',
 '그럼 1% 이율입니까?',
 '대출기간은 어떻게 됩니까?',
 '5월에 퇴사 했는데 연말정산 해야합니까?'

In [ ]:
question = list(s.split() for s in question)
ques = list(filter(None, question))

In [ ]:
sentence = list(' '.join(a) for a in ques)

In [ ]:
sentence

['지방세를 내려면 어떻게 해야됩니까?',
 '은행을 직접방문해도 됩니까?',
 '버스로 가는 방법도 있습니까?',
 '다른 납부방법도 있습니까?',
 '사이트 주소가 어떻게 됩니까?',
 '다른곳에서는 납부할수 없습니까?',
 '지방세는 조회할 수 있습니까?',
 '어떻게 납부합니까?',
 '서울시주최 페스티벌 예매해놨는데 예정대로 진행됩니까?',
 '코로나로 다른 축제들은 취소됐는데 이건 취소 안됩니까?',
 '환불할수 있습니까?',
 '환불수수료 있습니까?',
 '코로나때문에 못가는건데도 수수료 내야합니까?',
 '참석했다가 코로나 걸리면 보상해줍니까?',
 '그래도 걸리면 보상해줍니까?',
 '확산세가 더 강해지면 취소될 가능성 있습니까?',
 '그럼 전액 환불되는겁니까?',
 '청년저축계좌 지금 신청할 수 있습니까?',
 '그럼 다른 정책은 없습니까?',
 '그건 뭡니까?',
 '서울시민 대상입니까?',
 '얼마를 추가 적립해줍니까?',
 '모집인원은 어떻게 됩니까?',
 '신청기간은 언제까지입니까?',
 '네 주소지가 서울이면 되는겁니까?',
 '그럼 총 어느정도 혜택을 받을 수 있습니까?',
 '3년동안 계속 10만원씩 해야합니까?',
 '일도 쉬면 안됩니까?',
 '또 다른 조건 있습니까?',
 '저희가 작년에 결혼했는데 신혼부부 조건이 맞습니까?',
 '대출 상품이 뭐가 있습니까?',
 '어떤 은행에서 받을 수 있습니까?',
 '버팀목 대출 받으려면 조건이 있습니까?',
 '서울소재 주택만 가능한겁니까?',
 '대출한도는 어떻게 됩니까?',
 '100% 다 대출받을 수 있습니까?',
 '그럼 나머지 10%에 대해서는 그냥 대출을 받을 수 있습니까?',
 '그럼 그 한정된 곳에서 집을 찾아야합니까?',
 '대출 한도는 어떻게 됩니까?',
 '금리는 얼마입니까?',
 '그렇게 낮은편이 아닌데 다른 혜택은 없습니까?',
 '그게 뭡니까?',
 '그럼 1% 이율입니까?',
 '대출기간은 어떻게 됩니까?',
 '5월에 퇴사 했는데 연말정산 해야합니까?'

In [ ]:
# 질문 목록
new_questions = [
    '여권 재발급 신청 방법을 알려줘',
    '전입 신고가 인터넷으로 가능한가요?',
    '지방세 환급금을 어디서 신청하나요?'
]

In [ ]:
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1, 2),
    lowercase=False
)


In [ ]:
X = vectorizer.fit_transform(sentence)

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
X.toarray().shape

(19050, 43933)

In [ ]:
query_vec = vectorizer.transform(new_questions)

In [ ]:
sims = cosine_similarity(query_vec, X)
sims

array([[0.        , 0.01631305, 0.04717463, ..., 0.        , 0.        ,
        0.        ],
       [0.00830794, 0.00844649, 0.02025835, ..., 0.0462272 , 0.02902143,
        0.0280113 ],
       [0.07180571, 0.02491219, 0.00210336, ..., 0.00185799, 0.00177504,
        0.00171326]], shape=(3, 19050))

In [435]:
rank = sims.argsort()[::-1]

In [436]:
rank

array([[22240, 22225, 22226, ...,  3106, 15767,  8675],
       [22240, 22225, 22226, ..., 21078,  6123, 17908],
       [22240, 22225, 22226, ...,  3207, 13642,  4615]], shape=(3, 22245))

In [ ]:
data

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,고객,7,지방세납부,,Q,은행을 직접방문해도 됩니까?,,,방문납부도 가능합니다.,은행,은행/공공기관,"은행,공공기관"
2,다산콜센터,일반행정 문의,B2240,고객,13,지방세납부,,Q,버스로 가는 방법도 있습니까?,,,도보로 가시는게 빠를 것 같습니다.,버스,버스/교통수단,"버스,교통수단"
3,다산콜센터,일반행정 문의,B2240,고객,15,지방세납부,,Q,다른 납부방법도 있습니까?,,,위택스 사이트에서 납부하실수 있습니다.,,납부방법/지방세,
4,다산콜센터,일반행정 문의,B2240,고객,17,지방세납부,,Q,사이트 주소가 어떻게 됩니까?,,,www.wetax.go.kr 입니다.,사이트,납부방법/지방세/ 위텍스/ 납부,"사이트,납부"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19048,다산콜센터,대중교통 안내,B25056,고객,17,시내버스 시간,,Q,그럼 여수 시내버스 막차는 몇 시 인가요?,,,노선마다 다르지만 여수 시내버스는 가장 늦는게 23시45분 입니다.,"여수, 시내, 버스, 차, 시","여수/지역, 버스/대중교통, 차/자동차, 시/시간","시내,시간"
19049,다산콜센터,대중교통 안내,B25057,고객,7,지하철 여부,,Q,혹시 평택에 지하철 있나요?,,,네 잠시만요.,"평택, 지하철","평택/지역, 지하철/대중교통","지하철,대중교통"
19050,다산콜센터,대중교통 안내,B25058,고객,5,시내버스 시간,,Q,익산 시내버스 첫차가 몇 시 인가요?,,,노선마다 다르지만 익산 시내버스는 가장 빠른게 05시00분 입니다.,"익산, 시내, 버스, 차, 시","익산/지역, 버스/대중교통, 차/자동차, 시/시간","시내,시간"
19051,다산콜센터,대중교통 안내,B25058,고객,11,시내버스 시간,,Q,그럼 익산 시내버스 막차는 몇 시 인가요?,,,많이들 궁금해 하더라구요.,"익산, 시내, 버스, 차, 시","익산/지역, 버스/대중교통, 차/자동차, 시/시간","시내,시간"


In [438]:
for idx, sim in enumerate(sims):
    # 질문
    question = new_questions[idx]
    print('유저의 질문 :', question)
    # sim 데이터에서 내림차순정렬을 한 인덱스의 목록
    sim_idxs = sim.argsort()[::-1]
    for i in sim_idxs[:2]:
        # i :유저의 질문에 가장 유사한 질문의 인덱스
        print(f'유사도 : {round(sim[i], 3)},  유사 질문 : {data.loc[i, '고객질문(요청)']}, 답변 : {data.loc[i, '상담사답변']}')

유저의 질문 : 여권 재발급 신청 방법을 알려줘
유사도 : 0.477,  유사 질문 : 4615    신청해야되는거예요?
4615              
Name: 고객질문(요청), dtype: object, 답변 : 4615                                    
4615    아니요. 서울역에서 하차하셔서 지하철 환승하셔야 합니다. 
Name: 상담사답변, dtype: object
유사도 : 0.475,  유사 질문 : 13642      이용금액은 어떻게 되나요?
13642    가격은 언제가 더 저렴한가요?
Name: 고객질문(요청), dtype: object, 답변 : 13642    
13642    
Name: 상담사답변, dtype: object
유저의 질문 : 전입 신고가 인터넷으로 가능한가요?
유사도 : 0.822,  유사 질문 : 17908    
17908    
Name: 고객질문(요청), dtype: object, 답변 : 17908    
17908    
Name: 상담사답변, dtype: object
유사도 : 0.698,  유사 질문 : 6123                 
6123    터미널 앞쪽까지 가나요?
Name: 고객질문(요청), dtype: object, 답변 : 6123    아니요, 교육별로 모집기간과 교육기간이 상이하며 현재는 모든 교육이 신청마감 됐습니다
6123                                                   
Name: 상담사답변, dtype: object
유저의 질문 : 지방세 환급금을 어디서 신청하나요?
유사도 : 0.613,  유사 질문 : 8675    제가 지금 지방에 출장중이거든요.
8675                      
Name: 고객질문(요청), dtype: object, 답변 : 8675                               
8675    15KM정도 가시다 우측에 연기주유소가 있습니다.
Nam

In [ ]:
import numpy as np

In [ ]:
# TF-IDF 벡터화
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(data['고객질문(요청)'])

# 새 질문도 벡터화 (기존 vectorizer 기준으로 transform만!)
new_tfidf = vectorizer.transform(new_questions)

# 코사인 유사도 계산
similarities = cosine_similarity(new_tfidf, tfidf_matrix)

# 각 질문마다 유사도 높은 인덱스 순으로 정렬
rank = np.argsort(-similarities, axis=1)  # 내림차순 정렬

# 결과 출력
for i, question in enumerate(new_questions):
    print(f"\n 새 질문 {i+1}: {question}")
    
    for idx in rank[i][:3]:  # 상위 3개 유사 질문
        sim = round(similarities[i][idx], 3)
        print(f"  유사도: {sim}")
        print(f"  기존 질문: {data.iloc[idx]['고객질문(요청)']}")
        print(f"  답변: {data.iloc[idx]['상담사답변']}\n")


 새 질문 1: 여권 재발급 신청 방법을 알려줘
  유사도: 0.438
  기존 질문: 청년수당 신청 방법을 알 수 있을까요?
  답변: 서울청년포털 홈페이지에서 가능합니다. 

  유사도: 0.423
  기존 질문: 여권 재발급 비용도 알 수있나요?
  답변: 복수여권53000원,일반여권 50000원입니다.

  유사도: 0.355
  기존 질문: 그럼 등수도 알려줘?
  답변: 등수는 확인할 수 없어요. 


 새 질문 2: 전입 신고가 인터넷으로 가능한가요?
  유사도: 0.691
  기존 질문: 인터넷으로 가능한가요?
  답변: 인터넷으로 신청 가능합니다.

  유사도: 0.569
  기존 질문: 이런 것도 신고가 가능한가요?
  답변: 네~~가능합니다.

  유사도: 0.513
  기존 질문: 네, 인터넷으로 예약 가능한가요?
  답변: 네, 가능합니다. 


 새 질문 3: 지방세 환급금을 어디서 신청하나요?
  유사도: 0.611
  기존 질문: 어디서 신청하나요
  답변: 광역알뜰교통카드 홈페이지에서 할 수 있습니다.

  유사도: 0.611
  기존 질문: 어디서 신청하나요?
  답변: 온라인청년센터 통해서 신청가능합니다!

  유사도: 0.484
  기존 질문: 어떻게 신청하나요?
  답변: 주민센터에서 접수받고 있습니다.



In [ ]:
for i in range(len(rank)):
    print(f"\n 기준 질문 {i} :", data.iloc[i]['고객질문(요청)'])
    
    for idx in rank[i][:3]:  # 상위 3개만 보기
        print(f"  → 유사 질문 index:{idx}, 내용: {data.iloc[idx]['고객질문(요청)']}")


 기준 질문 0 : 지방세를 내려면 어떻게 해야됩니까?
  → 유사 질문 index:13068, 내용: 센트럴에서 블루스퀘어까지 가깝나요?
  → 유사 질문 index:16220, 내용: 광화문 먼저갔다 가는게 났겟군요~
  → 유사 질문 index:15338, 내용: 택시로 가면 몇분걸리는데요?

 기준 질문 1 : 은행을 직접방문해도 됩니까?
  → 유사 질문 index:17591, 내용: 1회용 카드도 기본요금이 교통카드와 같나요?
  → 유사 질문 index:6305, 내용:  증명서 언제 부터 사용 가능 하죠?
  → 유사 질문 index:17228, 내용: 온라인 예매는 어떻게 하나요?

 기준 질문 2 : 버스로 가는 방법도 있습니까?
  → 유사 질문 index:19046, 내용: 군산 시내버스 첫차가 몇 시 인가요?
  → 유사 질문 index:19025, 내용: 혹시 성남에 지하철 있나요?
  → 유사 질문 index:19026, 내용: 영동 시내버스 첫차가 몇 시 인가요?


### 문제 2
- 고객질문의 데이터를 이용하여 카테고리를 분류하는 모델을 생성
    - 고객질문 데이터들을 이용하여 토큰화, 벡터화 작업(독립변수)
    - 카테고리 일반행정, 대중교통을 타켓 데이터(종속변수)
        - 카테고리 데이터를 LabelEncoder()를 이용하여 수치화 변환
    - SVC 모델을 이용하여 벡터화된 데이터와 카테고리 데이터를 이용하여 학습
    - new_questions의 카테고리를 확인

In [ ]:
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1, 2),
    lowercase=False
)

In [ ]:
X = vectorizer.fit_transform(questions)

In [ ]:
sentence

['지방세를 내려면 어떻게 해야됩니까?',
 '은행을 직접방문해도 됩니까?',
 '버스로 가는 방법도 있습니까?',
 '다른 납부방법도 있습니까?',
 '사이트 주소가 어떻게 됩니까?',
 '다른곳에서는 납부할수 없습니까?',
 '지방세는 조회할 수 있습니까?',
 '어떻게 납부합니까?',
 '서울시주최 페스티벌 예매해놨는데 예정대로 진행됩니까?',
 '코로나로 다른 축제들은 취소됐는데 이건 취소 안됩니까?',
 '환불할수 있습니까?',
 '환불수수료 있습니까?',
 '코로나때문에 못가는건데도 수수료 내야합니까?',
 '참석했다가 코로나 걸리면 보상해줍니까?',
 '그래도 걸리면 보상해줍니까?',
 '확산세가 더 강해지면 취소될 가능성 있습니까?',
 '그럼 전액 환불되는겁니까?',
 '청년저축계좌 지금 신청할 수 있습니까?',
 '그럼 다른 정책은 없습니까?',
 '그건 뭡니까?',
 '서울시민 대상입니까?',
 '얼마를 추가 적립해줍니까?',
 '모집인원은 어떻게 됩니까?',
 '신청기간은 언제까지입니까?',
 '네 주소지가 서울이면 되는겁니까?',
 '그럼 총 어느정도 혜택을 받을 수 있습니까?',
 '3년동안 계속 10만원씩 해야합니까?',
 '일도 쉬면 안됩니까?',
 '또 다른 조건 있습니까?',
 '저희가 작년에 결혼했는데 신혼부부 조건이 맞습니까?',
 '대출 상품이 뭐가 있습니까?',
 '어떤 은행에서 받을 수 있습니까?',
 '버팀목 대출 받으려면 조건이 있습니까?',
 '서울소재 주택만 가능한겁니까?',
 '대출한도는 어떻게 됩니까?',
 '100% 다 대출받을 수 있습니까?',
 '그럼 나머지 10%에 대해서는 그냥 대출을 받을 수 있습니까?',
 '그럼 그 한정된 곳에서 집을 찾아야합니까?',
 '대출 한도는 어떻게 됩니까?',
 '금리는 얼마입니까?',
 '그렇게 낮은편이 아닌데 다른 혜택은 없습니까?',
 '그게 뭡니까?',
 '그럼 1% 이율입니까?',
 '대출기간은 어떻게 됩니까?',
 '5월에 퇴사 했는데 연말정산 해야합니까?'

In [ ]:
df['고객질문(요청)']

0            지방세를 내려면 어떻게 해야됩니까?
1                               
2                  은행 어플에서도 됩니까?
3                               
4                               
                  ...           
38961                           
38962    그럼 순천 시내버스 막차는 몇 시 인가요?
38963                           
38964                           
38965                           
Name: 고객질문(요청), Length: 89302, dtype: object

In [ ]:
data['tokens'] = data['고객질문(요청)'].apply(lambda x: okt.morphs(str(x)))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
import pandas as pd

# ----------------------------------------------------

# ----------------------------------------------------
# 2️⃣ 독립변수(X), 종속변수(y) 분리
# ----------------------------------------------------
X = data['고객질문(요청)']
y = data['카테고리']

# ----------------------------------------------------
# 3️⃣ 카테고리 수치화 (Label Encoding)
# ----------------------------------------------------
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# ----------------------------------------------------
# 4️⃣ TF-IDF 벡터화
# ----------------------------------------------------
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(X)

# ----------------------------------------------------
# 5️⃣ SVC 모델 학습
# ----------------------------------------------------
model = SVC(kernel='linear', probability=True)
model.fit(X_tfidf, y_encoded)

# ----------------------------------------------------
# 6️⃣ 새 질문 예측
# ----------------------------------------------------
new_questions = [
    '여권 재발급 신청 방법을 알려줘',
    '전입 신고가 인터넷으로 가능한가요?',
    '지방세 환급금을 어디서 신청하나요?'
]

# 새 질문 벡터화
new_tfidf = vectorizer.transform(new_questions)

# 예측
pred_labels = model.predict(new_tfidf)
pred_categories = le.inverse_transform(pred_labels)

# ----------------------------------------------------
# 7️⃣ 결과 출력
# ----------------------------------------------------
for q, cat in zip(new_questions, pred_categories):
    print(f"🟢 질문: {q}")
    print(f"➡️ 예측된 카테고리: {cat}\n")

🟢 질문: 여권 재발급 신청 방법을 알려줘
➡️ 예측된 카테고리: 일반행정 문의

🟢 질문: 전입 신고가 인터넷으로 가능한가요?
➡️ 예측된 카테고리: 일반행정 문의

🟢 질문: 지방세 환급금을 어디서 신청하나요?
➡️ 예측된 카테고리: 일반행정 문의



In [432]:
data = pd.concat([df1, df2])

In [433]:
data

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까?,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
3,다산콜센터,일반행정 문의,B2240,상담사,4,,지방세납부,Q,,어떤 은행을 이용하고 계십니까?,,,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,5,지방세납부,,A,,,기업은행을 이용하고 있습니다.,,기업은행,기업은행/상호,"기업은행,상호"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38961,다산콜센터,대중교통 안내,B25059,상담사,16,,시내버스 시간,A,,,,네 질문 해 주세요.,질문,질문/문의,"질문,문의"
38962,다산콜센터,대중교통 안내,B25059,고객,17,시내버스 시간,,Q,그럼 순천 시내버스 막차는 몇 시 인가요?,,,,"순천, 시내, 버스, 차, 시","순천/지역, 버스/대중교통, 차/자동차, 시/시간","시내,시간"
38963,다산콜센터,대중교통 안내,B25059,상담사,18,,시내버스 시간,A,,,,노선마다 다르지만 순천 시내버스는 가장 늦는게 23시55분 입니다.,"노선, 순천, 시내, 버스, 시, 분","노선/차편, 순천/지역, 버스/대중교통, 시/시간, 분/시간","순천,시간"
38964,다산콜센터,대중교통 안내,B25059,고객,19,시내버스 시간,,A,,,아 그렇군요. 도움이 많이 됐습니다. 감사합니다.,,도움,도움/상담,"도움,상담"


In [434]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------
# 1️⃣ 데이터 예시
# ---------------------------------------------
# data = pd.read_csv('your_file.csv')
# 여기엔 최소한 아래 컬럼이 있어야 함:
# ['화자', '고객질문(요청)', '상담사답변']

# ---------------------------------------------
# 2️⃣ 고객 질문 → 바로 상담사 답변인 데이터만 필터
# ---------------------------------------------
filtered_data = []
for i in range(len(data) - 1):
    if data.iloc[i]['화자'] == '고객' and data.iloc[i + 1]['화자'] == '상담사':
        q = data.iloc[i]['고객질문(요청)']
        a = data.iloc[i + 1]['상담사답변']
        filtered_data.append([q, a])

filtered_df = pd.DataFrame(filtered_data, columns=['질문', '답변'])

print(f"필터링된 데이터 수: {len(filtered_df)}")

# ---------------------------------------------
# 3️⃣ 중복 질문 제거
# ---------------------------------------------
filtered_df = filtered_df.drop_duplicates(subset=['질문']).reset_index(drop=True)
print(f"중복 제거 후 데이터 수: {len(filtered_df)}")

# ---------------------------------------------
# 4️⃣ TF-IDF 벡터화 (질문만)
# ---------------------------------------------
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(filtered_df['질문'])

# ---------------------------------------------
# 5️⃣ 새로운 질문 목록 정의
# ---------------------------------------------
new_questions = [
    '여권 재발급 신청 방법을 알려줘',
    '전입 신고가 인터넷으로 가능한가요?',
    '지방세 환급금을 어디서 신청하나요?'
]

# 새 질문 벡터화
new_tfidf = vectorizer.transform(new_questions)

# ---------------------------------------------
# 6️⃣ 코사인 유사도 계산
# ---------------------------------------------
sims = cosine_similarity(new_tfidf, tfidf_matrix)
rank = sims.argsort(axis=1)[:, ::-1]  # 유사도 높은 순 정렬

# ---------------------------------------------
# 7️⃣ 유사 질문 & 답변 출력
# ---------------------------------------------
for i, q in enumerate(new_questions):
    print(f"\n🟢 새 질문: {q}")
    for j in rank[i][:3]:  # 상위 3개 유사 질문
        print(f"  🔹유사도: {round(sims[i][j], 3)}")
        print(f"  🔸유사 질문: {filtered_df.iloc[j]['질문']}")
        print(f"  💬답변: {filtered_df.iloc[j]['답변']}\n")

필터링된 데이터 수: 41248
중복 제거 후 데이터 수: 22245

🟢 새 질문: 여권 재발급 신청 방법을 알려줘
  🔹유사도: 0.477
  🔸유사 질문: 여권 재발급 받고 싶어요
  💬답변: 

  🔹유사도: 0.475
  🔸유사 질문: 여권 재발급 받으려고 하는데요.
  💬답변: 

  🔹유사도: 0.449
  🔸유사 질문: 안녕하세요 여권 재발급 문의 드려요.
  💬답변: 


🟢 새 질문: 전입 신고가 인터넷으로 가능한가요?
  🔹유사도: 0.822
  🔸유사 질문: 신고가 가능한가요?
  💬답변: 

  🔹유사도: 0.698
  🔸유사 질문: 인터넷으로 가능한가요?
  💬답변: 인터넷으로 신청 가능합니다.

  🔹유사도: 0.564
  🔸유사 질문: 이런 것도 신고가 가능한가요?
  💬답변: 네~~가능합니다.


🟢 새 질문: 지방세 환급금을 어디서 신청하나요?
  🔹유사도: 0.613
  🔸유사 질문: 어디서 신청하나요?
  💬답변: 온라인청년센터 통해서 신청가능합니다!

  🔹유사도: 0.613
  🔸유사 질문: 어디서 신청하나요
  💬답변: 광역알뜰교통카드 홈페이지에서 할 수 있습니다.

  🔹유사도: 0.486
  🔸유사 질문: 어떻게 신청하나요?
  💬답변: 

